In [17]:
import torch
import torch.nn as nn
import torchaudio.transforms as T


DEGRADATION_LEVELS = {
    0: "perfect",
    1: "good",
    2: "okay",
    3: "bad",
    4: "very_bad",
}

NUM_CLASSES = len(DEGRADATION_LEVELS)

SAMPLE_RATE = 16_000
N_MELS = 128
WIN_LENGTH_MS = 25
HOP_LENGTH_MS = 10
WIN_LENGTH = int(SAMPLE_RATE * WIN_LENGTH_MS / 1000)
HOP_LENGTH = int(SAMPLE_RATE * HOP_LENGTH_MS / 1000)
N_FFT = 1024


class MelFrontend(nn.Module):
    def __init__(
        self,
        sample_rate: int = SAMPLE_RATE,
        n_fft: int = N_FFT,
        win_length: int = WIN_LENGTH,
        hop_length: int = HOP_LENGTH,
        n_mels: int = N_MELS,
        f_min: float = 300.0,
        f_max: float = 3400.0,
    ) -> None:
        super().__init__()

        self.n_fft = n_fft
        self.win_length = win_length
        self.hop_length = hop_length

        self.amplitude_to_db = T.AmplitudeToDB(stype="power", top_db=80.0)

        mel_fb = T.MelSpectrogram(
            sample_rate=sample_rate,
            n_fft=n_fft,
            win_length=win_length,
            hop_length=hop_length,
            n_mels=n_mels,
            f_min=f_min,
            f_max=f_max,
        ).mel_scale.fb.cpu()

        self._mel_fb = mel_fb
        self._window = torch.hann_window(win_length).cpu()

    def forward(self, waveform: torch.Tensor) -> torch.Tensor:
        device = waveform.device
        waveform = waveform.cpu()

        stft = torch.stft(
            waveform,
            n_fft=self.n_fft,
            hop_length=self.hop_length,
            win_length=self.win_length,
            window=self._window,
            return_complex=True,
        )
        power = stft.abs() ** 2

        mel = torch.matmul(power.transpose(-1, -2), self._mel_fb).transpose(-1, -2)

        mel_db = self.amplitude_to_db(mel)

        return mel_db.unsqueeze(1).to(device)


class CNNBlock(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        pool_size: int = 2,
    ) -> None:
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=kernel_size,
                padding=kernel_size // 2,
                bias=False,
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=pool_size, stride=pool_size),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class VHFNoiseClassifier(nn.Module):
    def __init__(
        self,
        num_classes: int = NUM_CLASSES,
        dropout: float = 0.3,
        sample_rate: int = SAMPLE_RATE,
        n_mels: int = N_MELS,
    ) -> None:
        super().__init__()

        self.frontend = MelFrontend(sample_rate=sample_rate, n_mels=n_mels)

        self.cnn = nn.Sequential(
            CNNBlock(in_channels=1, out_channels=32),
            CNNBlock(in_channels=32, out_channels=64),
            CNNBlock(in_channels=64, out_channels=128),
        )

        self.gap = nn.AdaptiveAvgPool2d(output_size=(1, 1))

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(256, num_classes),
        )

        self.gate_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, 1),
            nn.Sigmoid(),
        )

        self._init_weights()

    def _init_weights(self) -> None:
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(
        self, waveform: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        x = self.frontend(waveform)

        x = self.cnn(x)

        x = self.gap(x)

        logits = self.classifier(x)
        gate = self.gate_head(x)

        probs = torch.softmax(logits, dim=-1)

        return logits, probs, gate

    @torch.no_grad()
    def predict(self, waveform: torch.Tensor) -> dict[str, torch.Tensor]:
        self.eval()
        logits, probs, gate = self(waveform)
        label_idx = probs.argmax(dim=-1)
        label_name = [DEGRADATION_LEVELS[i.item()] for i in label_idx]

        return {
            "label_idx": label_idx,
            "label_name": label_name,
            "probs": probs,
            "gate": gate,
        }

    @torch.no_grad()
    def predict_long(
        self,
        waveform: torch.Tensor,
        window_sec: float = 2.0,
        hop_sec: float = 1.0,
    ) -> dict:
        self.eval()

        device = next(self.parameters()).device
        window_samples = int(window_sec * SAMPLE_RATE)
        hop_samples = int(hop_sec * SAMPLE_RATE)
        total_samples = waveform.shape[-1]

        windows = []
        start = 0
        while start + window_samples <= total_samples:
            windows.append(waveform[start : start + window_samples])
            start += hop_samples

        if len(windows) == 0:
            pad = torch.zeros(window_samples, dtype=waveform.dtype)
            pad[:total_samples] = waveform
            windows = [pad]

        batch = torch.stack(windows).to(device)
        _, probs, gate = self(batch)

        window_labels = probs.argmax(dim=-1)

        final_label_idx = int(torch.mode(window_labels).values.item())

        mean_probs = probs.mean(dim=0)
        mean_gate = gate.mean(dim=0)

        return {
            "label_idx": final_label_idx,
            "label_name": DEGRADATION_LEVELS[final_label_idx],
            "probs": mean_probs,
            "gate": mean_gate,
            "window_labels": window_labels.tolist(),
            "window_gates": gate.squeeze(-1).tolist(),
            "n_windows": len(windows),
        }


class ClassifierLoss(nn.Module):
    def __init__(self, label_smoothing: float = 0.05) -> None:
        super().__init__()
        self.ce = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

    def forward(
        self,
        logits: torch.Tensor,
        labels: torch.Tensor,
        gate: torch.Tensor,
        lambda_gate: float = 0.1,
    ) -> torch.Tensor:
        ce_loss = self.ce(logits, labels)
        gate_target = labels.float() / (NUM_CLASSES - 1)
        gate_loss = torch.nn.functional.mse_loss(gate.squeeze(-1), gate_target)

        return ce_loss + lambda_gate * gate_loss

In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

model = VHFNoiseClassifier().to(device)


total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params     : {total:,}")
print(f"Trainable params : {trainable:,}\n")

# Dummy batch: 4 clips of 2 seconds at 16 kHz
batch_size = 4
clip_seconds = 2
waveform = torch.randn(batch_size, SAMPLE_RATE * clip_seconds).to(device)

logits, probs, gate = model(waveform)

print(f"Input waveform : {tuple(waveform.shape)}")
print(f"Logits         : {tuple(logits.shape)}")
print(f"Probs          : {tuple(probs.shape)}  (sum={probs.sum(-1).tolist()})")
print(f"Gate           : {tuple(gate.shape)}   (values={gate.squeeze().tolist()})\n")

# Test predict()
labels = torch.randint(0, NUM_CLASSES, (batch_size,)).to(device)
out = model.predict(waveform)
print("predict() output:")
for k, v in out.items():
    if isinstance(v, list):
        print(f"  {k}: {v}")
    else:
        print(f"  {k}: shape={tuple(v.shape)}, values={v.tolist()}")

# Test loss
criterion = ClassifierLoss()
loss = criterion(logits, labels, gate)
print(f"\nLoss (random labels): {loss.item():.4f}")
print("\n=== All checks passed ===")

# Test predict_long() — single 15s clip
long_clip = torch.randn(SAMPLE_RATE * 15).to(device)  # (240000,)
out_long = model.predict_long(long_clip, window_sec=2.0, hop_sec=1.0)
print("\npredict_long() on 15s clip (2s window, 1s hop):")
print(f"  n_windows    : {out_long['n_windows']}")
print(f"  window_labels: {out_long['window_labels']}")
print(f"  window_gates : {[f'{g:.3f}' for g in out_long['window_gates']]}")
print(f"  label_idx    : {out_long['label_idx']}  ({out_long['label_name']})")
print(f"  mean gate    : {out_long['gate'].item():.4f}")

# Test predict_long() — clip shorter than one window (edge case)
short_clip = torch.randn(SAMPLE_RATE * 1).to(device)  # (16000,) — 1s
out_short = model.predict_long(short_clip, window_sec=2.0, hop_sec=1.0)
print("\npredict_long() on 1s clip (shorter than window — edge case):")
print(f"  n_windows    : {out_short['n_windows']}  (padded)")
print(f"  label_name   : {out_short['label_name']}")
print("\n=== All checks passed ===")

=== VHFNoiseClassifier sanity check ===

Device: cpu
Total params     : 135,526
Trainable params : 135,526

Input waveform : (4, 32000)
Logits         : (4, 5)
Probs          : (4, 5)  (sum=[1.0000001192092896, 1.0, 1.0, 1.0])
Gate           : (4, 1)   (values=[0.6397039294242859, 0.6505492925643921, 0.6298784613609314, 0.6331676840782166])

predict() output:
  label_idx: shape=(4,), values=[0, 0, 0, 0]
  label_name: ['perfect', 'perfect', 'perfect', 'perfect']
  probs: shape=(4, 5), values=[[0.955520510673523, 0.0006160090561024845, 0.00025795953115448356, 0.04053749516606331, 0.0030679248739033937], [0.9555433392524719, 0.0006566130905412138, 0.0002708995307330042, 0.04042590409517288, 0.0031032245606184006], [0.9549394249916077, 0.0006392287905327976, 0.00026789112598635256, 0.041021112352609634, 0.003132348647341132], [0.9562332630157471, 0.0006250563892535865, 0.0002655182615853846, 0.0398082435131073, 0.0030680035706609488]]
  gate: shape=(4, 1), values=[[0.955225944519043], [0.9

In [19]:
import soundfile as sf
from pathlib import Path


def load_all_audio(dir_path: Path):
    for subdir in dir_path.iterdir():
        if subdir.is_dir():
            label_subdir = subdir.name
            for file in subdir.glob("*.wav"):
                waveform, sr = sf.read(file)
                yield label_subdir, torch.from_numpy(waveform).float(), sr

In [20]:
from paths import DATA_DIR

first = next(load_all_audio(DATA_DIR / "test_output" / "audio"))

In [21]:
data = list(load_all_audio(DATA_DIR / "test_output" / "audio"))

In [22]:
def cut_to_2_second_chunks(waveform: torch.Tensor, sr: int) -> torch.Tensor:
    chunk_size = sr * 2
    total = waveform.shape[0]
    n_chunks = total // chunk_size

    return waveform[: n_chunks * chunk_size].reshape(n_chunks, chunk_size)

In [23]:
chunks = [cut_to_2_second_chunks(entry[1], entry[2]) for entry in data]

In [24]:
len(chunks)

455

In [25]:
labels = [[entry[0]] * len(chunk) for entry, chunk in zip(data, chunks)]

In [26]:
flat_labels = [label for sublist in labels for label in sublist]

In [27]:
flat_chunks = torch.cat(chunks, dim=0)

In [28]:
print(f"Total chunks: {flat_chunks.shape[0]}")

Total chunks: 2745


In [29]:
print(f"Total labels: {len(flat_labels)}")

Total labels: 2745


In [30]:
model = VHFNoiseClassifier()

In [37]:
from tqdm import tqdm
from torch.utils.data import DataLoader, TensorDataset


def get_device() -> str:
    if torch.backends.mps.is_available():
        return "mps"
    elif torch.cuda.is_available():
        return "cuda"
    return "cpu"


def train(
    model: VHFNoiseClassifier,
    chunks: torch.Tensor,
    labels: torch.Tensor,
    val_chunks: torch.Tensor,
    val_labels: torch.Tensor,
    epochs: int = 10,
    batch_size: int = 32,
    lr: float = 1e-3,
    device: str = get_device(),
):
    print(f"Training on: {device}")
    model = model.to(device)

    train_dataset = TensorDataset(chunks, labels)
    val_dataset = TensorDataset(val_chunks, val_labels)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = ClassifierLoss()

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        train_bar = tqdm(
            train_loader,
            desc=f"Epoch {epoch + 1:>3}/{epochs} [train]",
            leave=True,
        )

        for batch_chunks, batch_labels in train_bar:
            batch_chunks = batch_chunks.to(device)
            batch_labels = batch_labels.to(device)

            optimizer.zero_grad()
            logits, probs, gate = model(batch_chunks)
            loss = criterion(logits, batch_labels, gate)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * len(batch_labels)
            train_correct += (probs.argmax(dim=-1) == batch_labels).sum().item()
            train_total += len(batch_labels)

            train_bar.set_postfix(
                loss=f"{train_loss / train_total:.4f}",
                acc=f"{train_correct / train_total * 100:.1f}%",
            )

        train_bar.close()

        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            val_bar = tqdm(
                val_loader,
                desc=f"Epoch {epoch + 1:>3}/{epochs}  [val] ",
                leave=True,
            )

            for batch_chunks, batch_labels in val_bar:
                batch_chunks = batch_chunks.to(device)
                batch_labels = batch_labels.to(device)

                logits, probs, gate = model(batch_chunks)
                loss = criterion(logits, batch_labels, gate)

                val_loss += loss.item() * len(batch_labels)
                val_correct += (probs.argmax(dim=-1) == batch_labels).sum().item()
                val_total += len(batch_labels)

                val_bar.set_postfix(
                    loss=f"{val_loss / val_total:.4f}",
                    acc=f"{val_correct / val_total * 100:.1f}%",
                )

            val_bar.close()

    return model

In [ ]:
label_to_idx = {v: k for k, v in DEGRADATION_LEVELS.items()}
# {"perfect": 0, "good": 1, "okay": 2, "bad": 3, "very_bad": 4}

int_labels = torch.tensor([label_to_idx[l] for l in flat_labels], dtype=torch.long)

In [39]:
from sklearn.model_selection import train_test_split

# Stratified 80/20 split
train_idx, val_idx = train_test_split(
    range(len(flat_chunks)),
    test_size=0.2,
    stratify=int_labels,
    random_state=42,
)

train_chunks = flat_chunks[train_idx]
train_labels = int_labels[train_idx]

val_chunks = flat_chunks[val_idx]
val_labels = int_labels[val_idx]

model = train(
    model,
    train_chunks,
    train_labels,
    val_chunks,
    val_labels,
    epochs=5,
    batch_size=64,
)

Training on: mps


Epoch   5/5  [val] : 100%|██████████| 9/9 [00:00<00:00, 22.16it/s, acc=90.5%, loss=0.4207]
